In [ ]:
# --- CELL 1: AGGRESSIVE SETUP (FIX BINARY ERROR) ---
import os
import time

print("⚙️ Cleaning environment...")


libs_to_remove = "numpy peft diffusers transformers accelerate huggingface_hub controlnet_aux safetensors tokenizers scipy scikit-image"
os.system(f"pip uninstall -y {libs_to_remove}")

print("🔧 Installing compatible stack...")
os.system("pip install \"numpy<2.0\" scipy scikit-image diffusers==0.27.2 transformers==4.38.2 accelerate==0.27.2 huggingface_hub==0.23.0 controlnet_aux==0.0.7 safetensors ultralytics sahi tokenizers==0.15.2")

print("\nINSTALLATION DONE.")

In [2]:
# --- CELL 2: FIRE GENERATION (FORCE OVERWRITE) ---
# This script generates fire effects on EVERY image.
# IT WILL OVERWRITE EXISTING FILES to ensure generation happens.

import os
import glob
import random
import shutil
import gc
import torch
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageEnhance, ImageDraw, UnidentifiedImageError
from tqdm.notebook import tqdm
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel

# ==============================================================================
# 1. CONFIGURATION 
# ==============================================================================


INPUT_IMAGES_DIR = "/kaggle/input/dataset-test/stable"     
INPUT_LABELS_DIR = "/kaggle/input/labels-test/labels"  


OUTPUT_ROOT = "/kaggle/working/fire_dataset_single"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Batch Processing Settings
BATCH_SIZE = 30 

# Generation Hyperparameters
STRENGTH = 0.90      
CN_SCALE = 0.70      
GUIDANCE = 12.0      

PROMPTS = [
    "massive forest fire, flames climbing up the original tree trunks, burning branches, forest floor covered in fire and embers, thick smoke, intense heat, realistic texture, cinematic lighting, 8k",
    "raging wildfire inside a forest, trees are burning but standing, ground vegetation on fire, heavy smoke, orange glow, dramatic shadows, photorealistic",
    "apocalyptic fire in the woods, burning bark, bright flames surrounding the area, dark ash ground, volumetric lighting from the fire"
]

NEGATIVE_PROMPT = "lava lake, volcano, magma ground, aerial view, destroyed landscape, flat ground, cartoon, blurry, melting objects, disappearing trees, new trees, forest density increase"

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def create_mask(image_pil, boxes):
    w, h = image_pil.size
    mask = Image.new("L", (w, h), 255) 
    draw = ImageDraw.Draw(mask)
    for box in boxes:
        xc, yc, bw, bh = box[-4:]
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        draw.rectangle([x1, y1, x2, y2], fill=0) 
    return mask.filter(ImageFilter.GaussianBlur(10))

def read_yolo_labels(txt_path):
    boxes = []
    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            for line in f:
                try:
                    boxes.append(list(map(float, line.strip().split())))
                except ValueError:
                    continue
    return boxes

# ==============================================================================
# 3. MODEL INITIALIZATION
# ==============================================================================

print("⚙️ Loading Generation Models...")
cn = ControlNetModel.from_pretrained("lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", controlnet=cn, torch_dtype=torch.float16, safety_checker=None
).to("cuda")

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("Xformers enabled.")
except:
    print("Xformers failed, using normal mode.")

pipe.enable_vae_slicing()

# ==============================================================================
# 4. MAIN EXECUTION (FORCE RUN)
# ==============================================================================

print(f"\n Starting Generation on folder: {INPUT_IMAGES_DIR}")

total_success = 0
total_skipped = 0

all_images = glob.glob(os.path.join(INPUT_IMAGES_DIR, "*.jpg")) + \
             glob.glob(os.path.join(INPUT_IMAGES_DIR, "*.png"))
total_imgs = len(all_images)
print(f"   Found {total_imgs} images.")

# --- BATCH LOOP ---
for i in range(0, total_imgs, BATCH_SIZE):
    batch_num = (i // BATCH_SIZE) + 1
    current_batch = all_images[i : i + BATCH_SIZE]
    
    print(f"\n Processing Batch {batch_num} ({len(current_batch)} images)...")
    
    batch_output_files = []

    for img_path in tqdm(current_batch, desc=f"Batch {batch_num}"):
        filename = os.path.basename(img_path)
        try:
            label_name = os.path.splitext(filename)[0] + ".txt"
            save_path = os.path.join(OUTPUT_ROOT, f"fire_{filename}")

            if os.path.exists(save_path):
                    batch_output_files.append(save_path) 
                    total_success += 1
                    continue
                
            # Load Image
            try:
                img = Image.open(img_path)
                img.load()
                img = img.convert("RGB")
            except (UnidentifiedImageError, OSError, AttributeError):
                total_skipped += 1
                continue 

            # Load Labels
            lbl_path = os.path.join(INPUT_LABELS_DIR, label_name)
            boxes = read_yolo_labels(lbl_path)
            
            # Generate
            mask = create_mask(img, boxes)
            out = pipe(
                prompt=random.choice(PROMPTS), 
                negative_prompt=NEGATIVE_PROMPT,
                image=img, mask_image=mask, control_image=img,
                num_inference_steps=40, 
                strength=STRENGTH, controlnet_conditioning_scale=CN_SCALE,
                guidance_scale=GUIDANCE
            ).images[0]

            # Post-Process
            person_mask = ImageOps.invert(mask).filter(ImageFilter.GaussianBlur(8))
            fire_glow = out.filter(ImageFilter.GaussianBlur(40))
            tinted_person = ImageEnhance.Contrast(Image.blend(img, fire_glow, 0.4)).enhance(1.15)
            final = Image.blend(Image.composite(tinted_person, out, person_mask), out, 0.1)

            # Draw Smoke
            smoke_layer = Image.new("RGBA", final.size, (0,0,0,0))
            draw_smoke = ImageDraw.Draw(smoke_layer)
            for box in boxes:
                xc, yc, bw, bh = box[-4:]
                if (bw * final.width * bh * final.height) < (final.width * final.height * 0.002): continue
                x1 = int((xc - bw/2) * final.width) - 30
                y1 = int((yc - bh/2) * final.height) - 30
                x2 = int((xc + bw/2) * final.width) + 30
                y2 = int((yc + bh/2) * final.height) + 30
                draw_smoke.rectangle([x1, y1, x2, y2], fill=(30, 30, 30, 250))
            
            final = final.convert("RGBA")
            final = Image.alpha_composite(final, smoke_layer.filter(ImageFilter.GaussianBlur(60)))
            final = final.convert("RGB")
            
            final.save(save_path)
            batch_output_files.append(save_path)
            total_success += 1

        except Exception as e:
            print(f"Warning on {filename}: {e}")
            total_skipped += 1

    # Zipping
    if batch_output_files:
        zip_name = f"fire_batch_{batch_num}"
        print(f"Saving Backup: {zip_name}.zip ...")
        temp_zip_dir = os.path.join("/kaggle/working", zip_name)
        os.makedirs(temp_zip_dir, exist_ok=True)
        for fpath in batch_output_files:
            shutil.copy(fpath, temp_zip_dir)
        shutil.make_archive(os.path.join("/kaggle/working", zip_name), 'zip', temp_zip_dir)
    
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*40)
print("GENERATION COMPLETE")
print(f"   Success: {total_success}")
print(f"   Skipped: {total_skipped}")
print("="*40)

⚙️ Loading Generation Models...


unet/diffusion_pytorch_model.safetensors not found


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet_inpaint.StableDiffusionControlNetInpaintPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


⚠️ Xformers failed, using normal mode.

🔥 Starting Generation on folder: /kaggle/input/dataset-test/stable
   Found 25 images.

👉 Processing Batch 1 (25 images)...


...

  0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/36 [00:00<?, ?it/s]

💾 Saving Backup: fire_batch_1.zip ...

✅ GENERATION COMPLETE
   Success: 25
   Skipped: 0
